# IE643 - Week 6: export of the deployed student for the phone (`export-v1.1`)

**Deployed model:** `K10s1_fb2` (distill-v10b, pre-registered rule passed). **Probe:** probe-v2 on the same encoder.
CPU only (Accelerator: None). No GPU quota used.

## What this notebook produces (`/kaggle/working/app_bundle/`)
| file | content |
|---|---|
| `encoder.onnx` (fp32) | uint8 image `[1,512,512]` -> 384 px antialiased resize (as two fixed matrices, bit-for-bit the training `F.interpolate`) -> 0.5/0.5 normalisation -> ViT-Tiny -> connector -> resampler: `prefix [1,32,768]`; plus the probe-v2 logistic heads folded in: `probe_logits [1,14]` |
| `decoder_{int8,w4,f16w,fp32}.onnx` (only the shipped one stays in the bundle) | one DistilGPT2 step (T = 1) with a **static KV cache** `[6,1,12,S,64]`, a position input, and the vocabulary pruned to the tokens the model can emit. Static shapes = no empty tensors, no HF cache classes, predictable on ONNX Runtime Mobile |
| `vocab.json` | pruned index -> text piece, EOS index |
| `app_config.json` | shapes, decode length, flag thresholds (90 % VAL sensitivity) and per-flag TEST sensitivity/PPV, labels never named by the text, limits statement, headline metrics |
| `samples/` | 24 TEST images (512 px PNG) + GT impression + reference prediction, for the in-app gallery |
| `export_results.json`, `export_log.txt` | all checks below |

## Decoding scheme used in the app
Greedy, one position per call: positions 0..31 feed the 32 prefix vectors (the token input is ignored), position 32 feeds BOS (= EOS id), then each emitted token is fed back; stop at EOS or after `max_new` tokens (= the training `gen_max`). Causal attention makes this identical to the batched prefill of `model.greedy()`.

## Pre-registered acceptance checks (fixed before the first run, 2026-10-25)
| # | check | pass |
|---|---|---|
| A0 | torch fp32 CPU reference (`model.greedy`) vs the saved GPU fp16 `pred_test.csv` | reported only (probe-v1 saw 94 % identical; fp16 autocast vs fp32) |
| A1 | ONNX encoder vs torch | prefix max abs diff < 1e-3; probe logits max abs diff < 1e-3; flag decisions (90 % thresholds) identical on every TEST study |
| A2 | vocabulary pruning | 0 emitted tokens outside the kept set in the fp32 reference on VAL + TEST (then pruned argmax = full argmax at every step) |
| A3 | ONNX fp32 decoder vs torch reference | >= 99 % identical TEST reports |
| A4 | INT8 decoder (dynamic, per-channel, MatMul weights only; LayerNorm/softmax fp32) vs ONNX fp32 | paired TEST CE14 and CE5 micro-F1: 95 % CI lower bound of (int8 - fp32) > -0.02; distinct reports >= 90 % of fp32; hit-cap and repeated-trigram rates <= fp32 + 1 pp. **Pass -> app ships `decoder_int8.onnx`.** |
| A5 | 4-bit **weight-only** decoder (ORT `MatMulNBits`, block 32, symmetric; activations fp32) vs ONNX fp32 | same criteria as A4 |
| A6 | fp16 **weight storage** (Cast to fp32 at load; compute fp32) vs ONNX fp32 | same criteria as A4. Shipping order: int8 (A4) -> w4 (A5) -> f16w (A6) -> fp32 |

**A5 and A6 were added on 2026-10-25 after the export-v1 dry run and before the full run.** In that dry run (first 50 TEST rows) dynamic INT8 kept only 8 % of the reports identical to fp32 and doubled the distinct reports (16 -> 33); CE14 CI lower bound -0.052. Likely cause: dynamic INT8 quantises activations with one scale per tensor, and GPT-2's residual stream has a few very large outlier channels. A5 and A6 keep activations in fp32. (8-bit weight-only was tried first and dropped: ORT has no fast 8-bit kernel, about 6x slower per step than fp32 in the offline test; 4-bit has one.) A4 stays exactly as pre-registered and is decided on the full run.

export-v1.1 also: dry runs (`N_EVAL` set) use a seeded random subset instead of the first rows (the first 50 TEST rows had no CE5 positive); the check against the probe-v2 saved features now reads its `test_cls`/`test_mean`/`test_idx` arrays.

The encoder stays fp32 so the safety flags are exactly the probe-v2 flags.

## Inputs to attach
1. The **distill-v10b** commit output (holds `distill_runs/K10s1_fb2/best.pt`, `pred_val.csv`, `pred_test.csv`).
2. **IE643 Cleaning Step 7** (`reports_clean.csv`) and **IE643 Cleaning** (`images_512.npy`).
3. The **probe-v2** commit output (`probe_K10s1_fb2.json`; `probe_features_K10s1_fb2.npz` is used for an extra check if present).

Internet on (distilgpt2 tokenizer, CheXbert weights). Runtime ~45-70 min on Kaggle CPU with `N_EVAL = None`.


In [ ]:
import time
T0 = time.time()
EXPORT_VERSION = "export-v1.1"
RUN = "K10s1_fb2"              # deployed model (v10b rule passed)
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
N_EVAL = None                  # None = all VAL/TEST studies; e.g. 100 for a quick dry run (checks then marked "partial")
N_SAMPLES = 24                 # TEST images copied into the app bundle
OPSET = 17
APP_TARGET = "0.90"            # probe threshold set used in the app (pre-registered in v10_plan)
N_BOOT = 1000
SEED = 0
OFFLINE_TEST = False           # only for testing this notebook outside Kaggle (random weights, fake tokenizer / CheXbert)
print("export version:", EXPORT_VERSION, "| run:", RUN)


In [ ]:
MODEL_PY = r'''"""IE643 student: ViT-Tiny encoder -> connector -> DistilGPT2 decoder (+ distillation / clinical heads).

Two ways to condition the decoder on the image (cfg.mode):
  "xattn"  : cross-attention inside every DistilGPT2 block (CvT2DistilGPT2 style). Main model.
  "prefix" : a Perceiver-style resampler turns the patch tokens into `n_prefix` vectors that are
             prepended to the text embeddings (ClipCap / LLaVA style). Ablation. With
             set_stage("align") kept for the whole run this is the frozen-LM prefix-tuning ablation.

Conventions
  * BOS == EOS == 50256 (GPT-2 has no BOS). Student vocab = 50257, teacher ids >= 50257 are dropped in the loss.
  * forward() returns logits ALIGNED with labels: logits[:, j] predicts labels[:, j]
    (input = [BOS, t0..t_{n-1}], labels = [t0..t_{n-1}, EOS]). No internal shift anywhere.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel

EOS_ID = 50256
VOCAB = 50257


@dataclass
class StudentConfig:
    # encoder
    # Main setting: native-384 checkpoint (576 patch tokens, 24x24 grid -> clean 4x4 pooling).
    # 224 ablation: encoder_name="vit_tiny_patch16_224.augreg_in21k_ft_in1k", img_size=224.
    # Keep encoder_name/img_size identical to the NN-retrieval baseline you compare against.
    encoder_name: str = "vit_tiny_patch16_384.augreg_in21k_ft_in1k"
    img_size: int = 384
    pretrained_enc: bool = True
    # decoder
    decoder_name: str = "distilgpt2"
    pretrained_dec: bool = True
    mode: str = "xattn"            # "xattn" | "prefix"
    n_prefix: int = 32             # prefix mode: number of visual prefix tokens
    resampler_heads: int = 8
    # distillation targets (cached teacher features)
    d_grid: int = 1024             # siglip_grid4  [16, 1024]
    d_mean: int = 1024             # siglip_mean   [1024]
    d_proj: int = 2560             # proj_mean     [2560]
    grid_hw: int = 4
    # clinical head
    n_clin: int = 14
    diag_prompt: bool = False      # A10: head predictions -> 14 prompt tokens between the visual prefix and BOS (prefix mode only)
    drop: float = 0.1
    max_len: int = 192             # max text tokens incl. BOS. EDA: median impression 86 tok, p95 187 (pre-cleaning);
                                   # set from the p95 of ntok_final in reports_clean.csv before training.


class Connector(nn.Module):
    """Linear(D_v->H) -> GELU -> Linear(H->H) -> LayerNorm (LLaVA-1.5 style 2-layer MLP)."""

    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.LayerNorm(d_out)

    def forward(self, x):
        return self.norm(self.net(x))


class Resampler(nn.Module):
    """m learned queries cross-attend to the visual tokens -> [B, m, H] (Perceiver / Q-Former lite)."""

    def __init__(self, d: int, n_queries: int, n_heads: int, drop: float = 0.0):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(n_queries, d) * 0.02)
        self.ln_q, self.ln_kv, self.ln_ff = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, kv):
        q = self.queries.unsqueeze(0).expand(kv.size(0), -1, -1)
        kv = self.ln_kv(kv)
        x = q + self.attn(self.ln_q(q), kv, kv, need_weights=False)[0]
        return x + self.ff(self.ln_ff(x))


class XRayReportStudent(nn.Module):
    def __init__(self, cfg: StudentConfig):
        super().__init__()
        assert cfg.mode in ("xattn", "prefix")
        self.cfg = cfg

        # ---- encoder -------------------------------------------------------------------------
        self.encoder = timm.create_model(
            cfg.encoder_name, pretrained=cfg.pretrained_enc, num_classes=0, img_size=cfg.img_size
        )
        self.n_prefix_tok = getattr(self.encoder, "num_prefix_tokens", 1)  # CLS (+ reg tokens)
        d_v = self.encoder.num_features
        self.d_v = d_v
        self.enc_cfg = self.encoder.pretrained_cfg  # mean/std for preprocessing

        # ---- decoder -------------------------------------------------------------------------
        xattn = cfg.mode == "xattn"
        if cfg.pretrained_dec:
            self.decoder = GPT2LMHeadModel.from_pretrained(cfg.decoder_name, add_cross_attention=xattn)
        else:  # offline / unit-test path: distilgpt2 geometry, random init
            gcfg = GPT2Config(n_layer=6, n_head=12, n_embd=768, vocab_size=VOCAB, n_positions=1024,
                              add_cross_attention=xattn)
            self.decoder = GPT2LMHeadModel(gcfg)
        self.d_t = self.decoder.config.n_embd

        # ---- connector -----------------------------------------------------------------------
        self.connector = Connector(d_v, self.d_t)
        self.resampler = (
            Resampler(self.d_t, cfg.n_prefix, cfg.resampler_heads, cfg.drop) if cfg.mode == "prefix" else None
        )

        # ---- auxiliary heads (train only; dropped at export) --------------------------------
        self.grid_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_grid))
        self.mean_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_mean))
        self.proj_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_proj))
        self.clin_head = nn.Sequential(
            nn.LayerNorm(2 * d_v), nn.Dropout(cfg.drop), nn.Linear(2 * d_v, d_v), nn.GELU(),
            nn.Linear(d_v, cfg.n_clin),
        )

        # ---- diagnosis prompts (A10): token_c = LayerNorm(label_c + state[s_c]), s_c in {0 = not positive, 1 = positive} ---
        if cfg.diag_prompt:
            assert cfg.mode == "prefix", "diagnosis prompts are implemented for the prefix connector only"
            self.prompt_label = nn.Parameter(torch.randn(cfg.n_clin, self.d_t) * 0.02)
            self.prompt_state = nn.Parameter(torch.randn(2, self.d_t) * 0.02)
            self.prompt_ln = nn.LayerNorm(self.d_t)
            # per-label decision thresholds of the head, set on VAL by the worker; a buffer so they are saved with the weights
            self.register_buffer("prompt_thr", torch.full((cfg.n_clin,), 0.5))

    # ------------------------------------------------------------------------------------------
    def encode(self, images):
        """images [B,3,H,W] (already normalised) -> dict of visual tokens and head predictions."""
        feats = self.encoder.forward_features(images)              # [B, npt+N, D]
        cls = feats[:, 0]
        patches = feats[:, self.n_prefix_tok:]                     # [B, N, D]
        B, N, D = patches.shape
        g = int(math.isqrt(N))
        assert g * g == N, "patch tokens must form a square grid"
        mean = patches.mean(1)
        grid = F.adaptive_avg_pool2d(patches.transpose(1, 2).reshape(B, D, g, g), self.cfg.grid_hw)
        grid = grid.flatten(2).transpose(1, 2)                     # [B, 16, D] row-major
        memory = self.connector(feats)                             # [B, npt+N, H]
        if self.resampler is not None:
            memory = self.resampler(memory)                        # [B, m, H]
        return {"memory": memory, "cls": cls, "mean": mean, "grid": grid}

    def heads(self, enc):
        return {
            "grid_pred": self.grid_head(enc["grid"]),              # [B,16,d_grid]
            "mean_pred": self.mean_head(enc["mean"]),              # [B,d_mean]
            "proj_pred": self.proj_head(enc["mean"]),              # [B,d_proj]
            "clin_logits": self.clin_head(torch.cat([enc["cls"], enc["mean"]], -1)),  # [B,n_clin]
        }

    def clin_probs(self, enc):
        return torch.sigmoid(self.clin_head(torch.cat([enc["cls"], enc["mean"]], -1)).float())   # [B, n_clin] fp32

    def prompt_embeds(self, states):
        """states [B, n_clin] in {0,1} -> prompt tokens [B, n_clin, H]."""
        s = states.long().clamp(0, 1)
        return self.prompt_ln(self.prompt_label.unsqueeze(0) + self.prompt_state[s])

    def head_states(self, enc):
        return (self.clin_probs(enc).detach() >= self.prompt_thr).long()

    def forward(self, images, input_ids, attention_mask=None, with_heads=True, prompt_states=None, prompt_gt=None, p_gt=0.0):
        """prompt_states: explicit [B, n_clin] states (diag_prompt only); otherwise the head's own thresholded predictions,
        replaced per study by prompt_gt with probability p_gt (training)."""
        enc = self.encode(images)
        mem = enc["memory"]
        hd = self.heads(enc) if (with_heads or self.cfg.diag_prompt) else {}
        if self.cfg.diag_prompt:
            if prompt_states is None:
                prompt_states = (torch.sigmoid(hd["clin_logits"].detach().float()) >= self.prompt_thr).long()
                if prompt_gt is not None and p_gt > 0:
                    use_gt = torch.rand(prompt_states.size(0), 1, device=prompt_states.device) < p_gt
                    prompt_states = torch.where(use_gt, (prompt_gt > 0.5).long(), prompt_states)
            mem = torch.cat([mem, self.prompt_embeds(prompt_states).to(mem.dtype)], 1)
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=input_ids, attention_mask=attention_mask,
                               encoder_hidden_states=mem, use_cache=False)
            logits = out.logits
        else:
            m = mem.size(1)
            emb = torch.cat([mem, self.decoder.transformer.wte(input_ids)], 1)
            am = None
            if attention_mask is not None:
                am = torch.cat([attention_mask.new_ones(attention_mask.size(0), m), attention_mask], 1)
            out = self.decoder(inputs_embeds=emb, attention_mask=am, use_cache=False)
            logits = out.logits[:, m:]                              # drop prefix positions
        res = {"logits": logits}
        if with_heads:
            res.update(hd)
        return res

    # ------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, images, max_new_tokens=60, num_beams=3, no_repeat_ngram_size=3,
                 length_penalty=1.0, **kw):
        """Beam (or greedy if num_beams=1) decoding with KV cache. Returns list[list[int]] without BOS/EOS."""
        assert not self.cfg.diag_prompt, "use greedy() for diagnosis-prompt models"
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        B = mem.size(0)
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        common = dict(max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False,
                      no_repeat_ngram_size=no_repeat_ngram_size, length_penalty=length_penalty,
                      eos_token_id=EOS_ID, pad_token_id=EOS_ID, early_stopping=True, **kw)
        if self.cfg.mode == "xattn":
            ids = self.decoder.generate(input_ids=bos, encoder_hidden_states=mem, **common)
            seqs = ids[:, 1:]                                       # strip BOS
        else:
            emb = torch.cat([mem, self.decoder.transformer.wte(bos)], 1)
            seqs = self.decoder.generate(inputs_embeds=emb, **common)  # returns only new tokens
        out = []
        for s in seqs.tolist():
            if EOS_ID in s:
                s = s[: s.index(EOS_ID)]
            out.append(s)
        self.train(was_training)
        return out

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128):
        """Greedy decoding with the decoder's KV cache, written by hand so it does not depend on the
        transformers version's generate() handling of encoder_hidden_states / inputs_embeds.
        Returns list[list[int]] without BOS/EOS."""
        was_training = self.training
        self.eval()
        mem = self.encode(images)["memory"]
        B = mem.size(0)
        cur = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=cur, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(cur)], 1), use_cache=True)
        seqs = []
        done = torch.zeros(B, dtype=torch.bool, device=mem.device)
        for _ in range(max_new_tokens):
            nxt = out.logits[:, -1, :VOCAB].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            seqs.append(nxt)
            done |= nxt == EOS_ID
            if bool(done.all()):
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=out.past_key_values, use_cache=True)
            if self.cfg.mode == "xattn":
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
        res = []
        for s in torch.stack(seqs, 1).tolist():
            res.append(s[: s.index(EOS_ID)] if EOS_ID in s else s)
        self.train(was_training)
        return res

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128, prompt_states=None):
        """Greedy decoding with an explicit KV-cache loop (does not rely on HF generate(), whose handling of
        encoder_hidden_states for decoder-only GPT-2 differs across transformers versions).
        Returns (list[list[int]] without BOS/EOS, list[bool] hit_cap = no EOS within max_new_tokens)."""
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        if self.cfg.diag_prompt:
            st = prompt_states if prompt_states is not None else self.head_states(enc)
            mem = torch.cat([mem, self.prompt_embeds(st).to(mem.dtype)], 1)
        B, dev = mem.size(0), mem.device
        xattn = self.cfg.mode == "xattn"
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=dev)
        if xattn:
            out = self.decoder(input_ids=bos, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(bos)], 1), use_cache=True)
        past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        steps = []
        for step in range(max_new_tokens):
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            steps.append(nxt)
            done = done | (nxt == EOS_ID)
            if bool(done.all()) or step == max_new_tokens - 1:
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=past, use_cache=True)
            if xattn:
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
            past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        toks = torch.stack(steps, 1).tolist()
        seqs, capped = [], []
        for s in toks:
            if EOS_ID in s:
                seqs.append(s[: s.index(EOS_ID)]); capped.append(False)
            else:
                seqs.append(s); capped.append(True)
        self.train(was_training)
        return seqs, capped

    # ------------------------------------------------------------------------------------------
    def _is_pretrained(self, name: str) -> bool:
        if name.startswith("encoder."):
            return True
        return name.startswith("decoder.") and "crossattn" not in name and "ln_cross_attn" not in name

    def set_stage(self, stage: str):
        """'align': freeze pretrained encoder+decoder, train connector / cross-attn / resampler / heads.
        'full'  : train everything (use a lower LR on pretrained parts via param_groups)."""
        assert stage in ("align", "full")
        for n, p in self.named_parameters():
            p.requires_grad = True if stage == "full" else not self._is_pretrained(n)

    def param_groups(self, lr_new=1e-4, lr_pre=3e-5, wd=0.01):
        buckets = {(pre, dec): [] for pre in (True, False) for dec in (True, False)}
        for n, p in self.named_parameters():
            if p.requires_grad:
                buckets[(self._is_pretrained(n), p.ndim >= 2 and "wpe" not in n)].append(p)
        groups = []
        for (pre, dec), ps in buckets.items():
            if ps:
                groups.append({"params": ps, "lr": lr_pre if pre else lr_new, "weight_decay": wd if dec else 0.0})
        return groups

    def size_report(self):
        def cnt(m):
            return sum(p.numel() for p in m.parameters())
        wte = self.decoder.transformer.wte.weight.numel()
        r = {
            "encoder": cnt(self.encoder),
            "connector": cnt(self.connector) + (cnt(self.resampler) if self.resampler is not None else 0),
            "decoder_total": cnt(self.decoder),
            "decoder_token_embedding": wte,
            "train_only_heads": sum(cnt(m) for m in (self.grid_head, self.mean_head, self.proj_head, self.clin_head)),
        }
        r["deploy_total"] = r["encoder"] + r["connector"] + r["decoder_total"]
        if self.cfg.diag_prompt:   # the head and the prompt tokens are part of the deployed graph for A10
            r["clin_head_and_prompts_deployed"] = cnt(self.clin_head) + self.prompt_label.numel() + self.prompt_state.numel() + cnt(self.prompt_ln)
            r["deploy_total"] += r["clin_head_and_prompts_deployed"]
        r["deploy_total_after_vocab_prune_to_8k"] = r["deploy_total"] - wte + 8000 * self.d_t
        return r
'''

CHEXBERT_LABELER_PY = r'''"""CheXbert labeler (Smit et al. 2020) for scoring generated report text.

Same architecture and checkpoint as the `f1chexbert` package (StanfordAIMI/RRG_scorers/chexbert.pth) but:
  * strict weight loading (only the known `position_ids` buffer may differ between transformers versions),
  * batched inference,
  * returns the raw 4-way classes so both label conventions can be computed:
      head classes 0 blank, 1 positive, 2 negative, 3 uncertain  (No Finding head: 0 blank, 1 positive)
      "pos_only"  : positive = class 1; uncertain counts as negative  (matches the IE643 baselines)
      "rrg"       : positive = class 1 or 3 (uncertain counts as positive; f1chexbert / RRG-paper convention)
"""
import numpy as np
import torch
import torch.nn as nn

CHEXBERT_ORDER = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
                  "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other",
                  "Fracture", "Support Devices", "No Finding"]


class _Labeler(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        h = bert.config.hidden_size
        self.linear_heads = nn.ModuleList([nn.Linear(h, 4) for _ in range(13)] + [nn.Linear(h, 2)])

    def forward(self, ids, att):
        cls = self.bert(ids, attention_mask=att)[0][:, 0]
        return [hd(cls) for hd in self.linear_heads]


class CheXbert:
    def __init__(self, device="cpu", checkpoint=None, random_init=False):
        from transformers import AutoConfig, AutoModel, BertTokenizer
        self.device = torch.device(device)
        self.tok = BertTokenizer.from_pretrained("bert-base-uncased")
        self.model = _Labeler(AutoModel.from_config(AutoConfig.from_pretrained("bert-base-uncased")))
        if not random_init:
            if checkpoint is None:
                from huggingface_hub import hf_hub_download
                checkpoint = hf_hub_download("StanfordAIMI/RRG_scorers", "chexbert.pth")
            sd = torch.load(checkpoint, map_location="cpu", weights_only=False)
            sd = sd.get("model_state_dict", sd)
            sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
            res = self.model.load_state_dict(sd, strict=False)
            bad_missing = [k for k in res.missing_keys if not k.endswith("position_ids")]
            bad_unexp = [k for k in res.unexpected_keys if not k.endswith("position_ids")]
            assert not bad_missing and not bad_unexp, f"CheXbert weights mismatch: missing {bad_missing[:5]} unexpected {bad_unexp[:5]}"
        self.model.to(self.device).eval()

    @staticmethod
    def _clean(t):
        return " ".join(str(t).strip().replace("\n", " ").split())

    @torch.no_grad()
    def classes(self, texts, bs=64):
        """-> int array [N,14] of raw classes in CHEXBERT_ORDER."""
        out = []
        for s in range(0, len(texts), bs):
            chunk = [self._clean(t) for t in texts[s:s + bs]]
            # [CLS] + wordpieces + [SEP], truncated to 512 (same ids as the reference tokenize -> encode_plus path;
            # encode_plus was removed in transformers 5)
            enc = [self.tok(t, add_special_tokens=True, truncation=True, max_length=512)["input_ids"] if t else
                   [self.tok.cls_token_id, self.tok.sep_token_id] for t in chunk]
            n = max(len(e) for e in enc)
            ids = torch.zeros(len(enc), n, dtype=torch.long)
            att = torch.zeros(len(enc), n, dtype=torch.float)
            for i, e in enumerate(enc):
                ids[i, :len(e)] = torch.tensor(e); att[i, :len(e)] = 1
            logits = self.model(ids.to(self.device), att.to(self.device))
            out.append(torch.stack([l.argmax(1) for l in logits], 1).cpu())
        return torch.cat(out).numpy()

    @staticmethod
    def binary(cls, labels, convention="pos_only"):
        """raw classes [N,14] -> 0/1 matrix [N,len(labels)] in the order of `labels`."""
        J = [CHEXBERT_ORDER.index(c) for c in labels]
        c = cls[:, J]
        pos = (c == 1) if convention == "pos_only" else ((c == 1) | (c == 3))
        return pos.astype(np.int8)
'''
print('sources embedded')


In [ ]:
import os, sys, json, glob, re, subprocess, zlib, dataclasses
import numpy as np, pandas as pd

def _need(mod, pkg):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
_need("onnx", "onnx")
_need("onnxruntime", "onnxruntime")
_need("onnx_ir", "onnx-ir")          # needed by onnxruntime.quantization.matmul_nbits_quantizer (not on the Kaggle image)
import torch, torch.nn as nn, torch.nn.functional as F, timm, transformers, onnx, onnxruntime as ort
torch.set_grad_enabled(False)
torch.manual_seed(SEED)

SRC_DIR = f"{WORK}/export_src"
os.makedirs(SRC_DIR, exist_ok=True)
for n, s in {"model.py": MODEL_PY, "chexbert_labeler.py": CHEXBERT_LABELER_PY}.items():
    open(f"{SRC_DIR}/{n}", "w").write(s)
sys.path.insert(0, SRC_DIR)
from model import StudentConfig, XRayReportStudent, EOS_ID, VOCAB

BUNDLE = f"{WORK}/app_bundle"
os.makedirs(f"{BUNDLE}/samples", exist_ok=True)
LOG = open(f"{WORK}/export_log.txt", "w")
def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True); LOG.write(s + "\n"); LOG.flush()
RES = {"version": EXPORT_VERSION, "run": RUN, "n_eval": N_EVAL, "checks": {}}
log(f"python {sys.version.split()[0]} | torch {torch.__version__} | timm {timm.__version__} | transformers {transformers.__version__} "
    f"| onnx {onnx.__version__} | onnxruntime {ort.__version__} | cpu threads {torch.get_num_threads()}")
RES["env"] = {"torch": torch.__version__, "timm": timm.__version__, "transformers": transformers.__version__,
              "onnx": onnx.__version__, "onnxruntime": ort.__version__}

# ---------------------------------------------------------------- inputs
def one(pattern, what, filt=None):
    c = sorted(glob.glob(f"{INPUT_ROOT}/{pattern}", recursive=True))
    if filt:
        c = [p for p in c if filt(p)]
    assert len(c) >= 1, f"{what} not found ({pattern}). Attach it as an input."
    if len(c) > 1:
        log(f"note: {len(c)} candidates for {what}, using the last: {c}")
    return c[-1]

CKPT = one(f"**/distill_runs/{RUN}/best.pt", f"{RUN}/best.pt (distill-v10b output)")
RUNDIR = os.path.dirname(CKPT)
CSV = one("**/reports_clean.csv", "reports_clean.csv (IE643 Cleaning Step 7)",
          lambda p: {"impression_final", "use_for_training"} <= set(pd.read_csv(p, nrows=0).columns))
NPY = one("**/images_512.npy", "images_512.npy (IE643 Cleaning)")
PROBE = one(f"**/probe_{RUN}.json", f"probe_{RUN}.json (probe-v2 output)")
_pf = glob.glob(f"{INPUT_ROOT}/**/probe_features_{RUN}.npz", recursive=True)
PROBE_FEATS = _pf[-1] if _pf else None
log(f"checkpoint {CKPT}\ncsv {CSV}\nnpy {NPY}\nprobe {PROBE}\nprobe features {PROBE_FEATS}")

df = pd.read_csv(CSV, low_memory=False)
imgs = np.load(NPY, mmap_mode="r")
assert len(df) == imgs.shape[0] and (df["idx"].values == np.arange(len(df))).all(), "CSV and images_512.npy are not aligned"
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
U = df[df.use_for_training]
SPL = {s: U[U.subset == s].reset_index(drop=True) for s in ("train", "val", "test")}
# dry runs use a seeded random subset: the first rows of a split are not representative (export-v1 dry run: first 50 TEST rows had no CE5 positive)
EV = {s: (SPL[s] if N_EVAL is None else SPL[s].sample(min(N_EVAL, len(SPL[s])), random_state=SEED).sort_values("idx")).reset_index(drop=True)
      for s in ("val", "test")}
log("usable per split:", {s: len(d) for s, d in SPL.items()}, "| evaluated:", {s: len(d) for s, d in EV.items()})

LABELS = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
          "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"]
FIVE = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
PJ = json.load(open(PROBE))
assert PJ["labels"] == LABELS and PJ["encoder_run"] == RUN, (PJ["labels"], PJ["encoder_run"])

# ---------------------------------------------------------------- model (fp32, CPU, no downloads: geometry from the config, weights from best.pt)
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
_fields = {f.name for f in dataclasses.fields(StudentConfig)}
cfg = StudentConfig(**{k: v for k, v in ck["student_config"].items() if k in _fields})
cfg.pretrained_enc = cfg.pretrained_dec = False
assert cfg.mode == "prefix" and not cfg.diag_prompt, (cfg.mode, cfg.diag_prompt)
model = XRayReportStudent(cfg)
model.load_state_dict({k: (t.float() if t.is_floating_point() else t) for k, t in ck["state_dict"].items()}, strict=True)
model.eval()
IMG = cfg.img_size
MAX_NEW = int(ck["run_cfg"]["gen_max"])
P = cfg.n_prefix
S_CACHE = P + 1 + MAX_NEW          # prefix + BOS + fed-back tokens (the last emitted token is never fed)
MEAN = torch.tensor(model.enc_cfg["mean"]).view(1, 3, 1, 1)
STD = torch.tensor(model.enc_cfg["std"]).view(1, 3, 1, 1)
gc = model.decoder.config
NL, NH, DH = gc.n_layer, gc.n_head, gc.n_embd // gc.n_head
assert gc.activation_function in ("gelu_new", "gelu_pytorch_tanh"), gc.activation_function
log(f"model: {cfg.encoder_name} @ {IMG} px, prefix {P}, decoder {NL} layers x {NH} heads x {DH} | best {'step' if 'scst' in str(ck['run_cfg']) else 'epoch'} "
    f"{ck.get('best_epoch')} | gen_max {MAX_NEW} -> cache length {S_CACHE} | mean/std {model.enc_cfg['mean']}/{model.enc_cfg['std']}")
RES["model"] = {"encoder": cfg.encoder_name, "img": IMG, "n_prefix": P, "max_new": MAX_NEW, "cache_len": S_CACHE,
                "best": ck.get("best_epoch"), "size": model.size_report()}

# ---------------------------------------------------------------- tokenizer + CheXbert
if OFFLINE_TEST:
    class _Tok:
        def __call__(self, t):
            return {"input_ids": [zlib.crc32(w.encode()) % 50000 for w in t.split()]}
        def decode(self, s, skip_special_tokens=True):
            return " ".join(f"w{t}" for t in s if t != EOS_ID)
    tok = _Tok()
    class _CXB:
        def classes(self, texts, bs=64):
            return np.array([[(zlib.crc32(t.encode()) >> k) % 4 for k in range(14)] for t in texts])
    from chexbert_labeler import CheXbert
    cxb = _CXB()
else:
    from transformers import AutoTokenizer
    from chexbert_labeler import CheXbert
    tok = AutoTokenizer.from_pretrained("distilgpt2")
    cxb = CheXbert(device="cpu")
binary = CheXbert.binary
def decode(s):
    return tok.decode(s, skip_special_tokens=True).strip()

def batch_x(u8):
    """uint8 [B,512,512] -> normalised [B,3,IMG,IMG], exactly as prep(train=False) in distill_worker (CPU, fp32)."""
    x = torch.as_tensor(np.ascontiguousarray(u8)).float().div_(255).unsqueeze(1)
    x = F.interpolate(x, size=(IMG, IMG), mode="bilinear", antialias=True, align_corners=False)
    return (x.expand(-1, 3, -1, -1) - MEAN) / STD
log(f"setup done in {time.time() - T0:.0f} s")


In [ ]:
# ---- torch fp32 CPU reference decode (model.greedy, the decoding used in every evaluation) on VAL + TEST
t0 = time.time()
REF = {}
for s in ("val", "test"):
    rows = EV[s].idx.values
    seqs, caps = [], []
    for i in range(0, len(rows), 16):
        sq, cp = model.greedy(batch_x(imgs[rows[i:i + 16]]), max_new_tokens=MAX_NEW)
        seqs += sq; caps += cp
    REF[s] = {"seqs": seqs, "cap": np.array(caps), "text": [decode(q) for q in seqs]}
    log(f"reference {s}: {len(seqs)} studies, {time.time() - t0:.0f} s, hit-cap {np.mean(caps):.1%}, distinct {len(set(REF[s]['text']))}")

# A0 (reported only): agreement with the saved GPU fp16-autocast predictions of the same checkpoint
A0 = {}
for s in ("val", "test"):
    p = f"{RUNDIR}/pred_{s}.csv"
    if os.path.exists(p):
        sv = pd.read_csv(p).set_index("idx")["pred"].fillna("").astype(str)
        same = [sv.get(int(i), None) == t for i, t in zip(EV[s].idx.values, REF[s]["text"])]
        A0[s] = float(np.mean(same))
log(f"A0 (reported): fp32 CPU reference identical to the saved fp16 GPU predictions: {A0}")
RES["checks"]["A0_ref_vs_saved_fp16"] = A0

# A2: vocabulary pruning. kept set = tokens of the TRAIN impressions + tokens the reference emits on VAL (+ EOS);
# checked on TEST, which did not shape the set.
keep = {EOS_ID}
for t in SPL["train"].impression_final:
    keep.update(tok(t)["input_ids"])
for q in REF["val"]["seqs"]:
    keep.update(q)
out_test = [t for q in REF["test"]["seqs"] for t in q if t not in keep]
n_emit = sum(len(q) for q in REF["test"]["seqs"])
A2_PASS = len(out_test) == 0
log(f"A2 vocabulary pruning: kept {len(keep)} of {VOCAB} ids; TEST emitted tokens outside the set: {len(out_test)} of {n_emit} "
    f"-> {'PASS' if A2_PASS else 'FAIL (ship the full vocabulary)'}")
if not A2_PASS:
    log("   outside ids:", sorted(set(out_test))[:50])
KEEP = sorted(keep) if A2_PASS else list(range(VOCAB))
K = len(KEEP)
FULL2K = {t: i for i, t in enumerate(KEEP)}
EOS_K = FULL2K[EOS_ID]
RES["checks"]["A2_vocab_prune"] = {"kept": K, "test_emitted": n_emit, "test_outside": len(out_test), "pass": A2_PASS}


In [ ]:
# ---- encoder graph: uint8 512x512 -> prefix [1,P,768] + probe logits [1,14]
def onnx_export(mod, args, path, in_names, out_names, dyn=None):
    kw = dict(input_names=in_names, output_names=out_names, opset_version=OPSET, do_constant_folding=True, dynamic_axes=dyn)
    try:
        torch.onnx.export(mod, args, path, dynamo=False, **kw)
    except TypeError:                                   # older torch without the dynamo argument
        torch.onnx.export(mod, args, path, **kw)
    onnx.checker.check_model(path)
    log(f"exported {os.path.basename(path)}: {os.path.getsize(path) / 2**20:.1f} MB")

def ort_session(path, threads=4):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

# the 512 -> IMG antialiased bilinear resize is separable and linear, so it equals R^T X R with a fixed [512, IMG] matrix R
R = F.interpolate(torch.eye(512)[None, None], size=(512, IMG), mode="bilinear", antialias=True, align_corners=False)[0, 0].contiguous()
_x = torch.rand(2, 512, 512)
_d = (R.T @ _x @ R - F.interpolate(_x[:, None], size=(IMG, IMG), mode="bilinear", antialias=True, align_corners=False)[:, 0]).abs().max()
log(f"resize as matrices vs F.interpolate: max abs diff {float(_d):.2e}")
assert _d < 1e-5

PW = torch.tensor(PJ["W"], dtype=torch.float32)           # [14, 384]
PB = torch.tensor(PJ["b"], dtype=torch.float32)
PM = torch.tensor(PJ["mean"], dtype=torch.float32)
PS = torch.tensor(PJ["std"], dtype=torch.float32)
THR = np.array(PJ["threshold"][APP_TARGET], dtype=np.float32)
assert PW.shape == (14, 2 * model.d_v), PW.shape

class EncoderExport(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer("R", R.clone()); self.register_buffer("RT", R.T.contiguous())
        self.register_buffer("mu", MEAN.clone()); self.register_buffer("sd", STD.clone())
        self.register_buffer("pw", PW.T.contiguous()); self.register_buffer("pb", PB.clone())
        self.register_buffer("pm", PM.clone()); self.register_buffer("ps", PS.clone())

    def forward(self, image):                               # image: uint8 [1,512,512]
        x = image.float() / 255.0
        x = self.RT @ x @ self.R                            # [1,IMG,IMG]
        x = (x.unsqueeze(1).expand(-1, 3, -1, -1) - self.mu) / self.sd
        e = self.m.encode(x)
        feat = torch.cat([e["cls"], e["mean"]], -1)         # probe-v2 feature set "clsmean"
        logits = ((feat - self.pm) / self.ps) @ self.pw + self.pb
        return e["memory"], logits, feat

encx = EncoderExport(model).eval()
ENC_PATH = f"{BUNDLE}/encoder.onnx"
_img0 = torch.as_tensor(np.ascontiguousarray(imgs[EV["test"].idx.values[:1]]))
onnx_export(encx, (_img0,), ENC_PATH, ["image"], ["prefix", "probe_logits", "features"])
enc_sess = ort_session(ENC_PATH)

# A1: ONNX encoder vs torch on every evaluated TEST study
t0 = time.time()
d_pre = d_log = 0.0
flag_mismatch = 0
ENC_PREFIX, ENC_FEAT, ENC_LOGITS = {}, [], []
for i in EV["test"].idx.values:
    u8 = np.ascontiguousarray(imgs[i:i + 1])
    pre, lg, ft = enc_sess.run(None, {"image": u8})
    e = model.encode(batch_x(u8))
    tf = torch.cat([e["cls"], e["mean"]], -1)
    tl = (((tf - PM) / PS) @ PW.T + PB).numpy()
    d_pre = max(d_pre, float(np.abs(pre - e["memory"].numpy()).max()))
    d_log = max(d_log, float(np.abs(lg - tl).max()))
    flag_mismatch += int(((lg >= THR) != (tl >= THR)).any())
    ENC_PREFIX[int(i)] = pre; ENC_FEAT.append(ft[0]); ENC_LOGITS.append(lg[0])
ENC_MS = (time.time() - t0) / len(EV["test"]) * 1000
A1_PASS = d_pre < 1e-3 and d_log < 1e-3 and flag_mismatch == 0
log(f"A1 encoder: prefix max abs diff {d_pre:.2e}, probe-logit max abs diff {d_log:.2e}, studies with any flag decision changed "
    f"{flag_mismatch}/{len(EV['test'])} -> {'PASS' if A1_PASS else 'FAIL'}  (ORT + torch: {ENC_MS:.0f} ms/study)")
A1 = {"prefix_maxdiff": d_pre, "logit_maxdiff": d_log, "flag_mismatch_studies": flag_mismatch, "pass": A1_PASS}

# extra (reported): features vs the probe-v2 saved features, if that output is attached
if PROBE_FEATS:
    try:
        z = np.load(PROBE_FEATS, allow_pickle=True)
        log("probe features npz keys:", {k: z[k].shape for k in z.files})
        if {"test_cls", "test_mean", "test_idx"} <= set(z.files):
            zf = np.concatenate([z["test_cls"], z["test_mean"]], 1)           # probe-v2 "clsmean" = concat(cls, mean)
            pos = {int(v): j for j, v in enumerate(z["test_idx"])}
            ok = [(pos[int(i)], j) for j, i in enumerate(EV["test"].idx.values) if int(i) in pos]
            dd = max(float(np.abs(zf[a] - ENC_FEAT[b]).max()) for a, b in ok)
            fl = sum(int(((((zf[a] - PM.numpy()) / PS.numpy()) @ PW.numpy().T + PB.numpy() >= THR) != (ENC_LOGITS[b] >= THR)).any()) for a, b in ok)
            A1["vs_probe_v2_saved_features_maxdiff"] = dd
            A1["vs_probe_v2_flag_mismatch_studies"] = fl
            log(f"   features vs probe-v2 saved ({len(ok)} studies): max abs diff {dd:.2e}; studies whose flags differ {fl}")
        else:
            log("   (could not identify feature / idx arrays; skipped)")
    except Exception as ex:
        log(f"   probe feature comparison skipped: {ex!r}")
RES["checks"]["A1_encoder"] = A1


In [ ]:
# ---- decoder graph: one DistilGPT2 position per call, static KV cache, pruned vocabulary
dec = model.decoder
tr = dec.transformer
EPS = gc.layer_norm_epsilon
import math

def _w(lin, d_in):
    """GPT-2 Conv1D stores [in, out]; nn.Linear stores [out, in]. Return [in, out]."""
    w = lin.weight.detach()
    w = w if hasattr(lin, "nf") else w.T              # transformers Conv1D has .nf/.nx; square c_proj makes a shape test ambiguous
    assert w.shape[0] == d_in, (type(lin), tuple(w.shape))
    return w.contiguous()

class DecoderStep(nn.Module):
    def __init__(self, keep):
        super().__init__()
        H = gc.n_embd
        self.register_buffer("wte", tr.wte.weight.detach()[torch.tensor(keep)].contiguous())   # [K, H]
        self.register_buffer("wpe", tr.wpe.weight.detach()[:S_CACHE].contiguous())
        self.layers = []
        for l, b in enumerate(tr.h):
            for n, t in {"ln1w": b.ln_1.weight, "ln1b": b.ln_1.bias, "qkvw": _w(b.attn.c_attn, H), "qkvb": b.attn.c_attn.bias,
                         "aow": _w(b.attn.c_proj, H), "aob": b.attn.c_proj.bias, "ln2w": b.ln_2.weight, "ln2b": b.ln_2.bias,
                         "fcw": _w(b.mlp.c_fc, H), "fcb": b.mlp.c_fc.bias, "pjw": _w(b.mlp.c_proj, 4 * H), "pjb": b.mlp.c_proj.bias}.items():
                self.register_buffer(f"l{l}_{n}", t.detach().clone())
        self.register_buffer("lnfw", tr.ln_f.weight.detach().clone()); self.register_buffer("lnfb", tr.ln_f.bias.detach().clone())
        self.register_buffer("ar", torch.arange(S_CACHE))

    def g(self, l, n):
        return getattr(self, f"l{l}_{n}")

    def forward(self, prefix, token, pos, k_cache, v_cache):
        # prefix [1,P,H] f32 | token [1] i64 (pruned index; ignored while pos < P) | pos [1] i64 | caches [NL,1,NH,S,DH]
        H = gc.n_embd
        e_pre = prefix[0].index_select(0, pos.clamp(max=P - 1))          # [1,H]
        e_tok = self.wte.index_select(0, token)                            # [1,H]
        x = torch.where((pos < P).view(1, 1), e_pre, e_tok) + self.wpe.index_select(0, pos)
        x = x.view(1, 1, H)
        write = (self.ar == pos).view(1, 1, S_CACHE, 1)
        attend = (self.ar <= pos).view(1, 1, 1, S_CACHE)
        ks, vs = [], []
        for l in range(NL):
            h = F.layer_norm(x, (H,), self.g(l, "ln1w"), self.g(l, "ln1b"), EPS)
            qkv = h @ self.g(l, "qkvw") + self.g(l, "qkvb")
            q, k, v = qkv.split(H, -1)
            q = q.view(1, 1, NH, DH).transpose(1, 2)                        # [1,NH,1,DH]
            k = k.view(1, 1, NH, DH).transpose(1, 2)
            v = v.view(1, 1, NH, DH).transpose(1, 2)
            kc = torch.where(write, k, k_cache[l])                          # [1,NH,S,DH]
            vc = torch.where(write, v, v_cache[l])
            a = (q @ kc.transpose(-1, -2)) * (1.0 / math.sqrt(DH))          # [1,NH,1,S]
            a = torch.where(attend, a, torch.full_like(a, -1e9)).softmax(-1)
            o = (a @ vc).transpose(1, 2).reshape(1, 1, H)
            x = x + o @ self.g(l, "aow") + self.g(l, "aob")
            h = F.layer_norm(x, (H,), self.g(l, "ln2w"), self.g(l, "ln2b"), EPS)
            h = F.gelu(h @ self.g(l, "fcw") + self.g(l, "fcb"), approximate="tanh")
            x = x + h @ self.g(l, "pjw") + self.g(l, "pjb")
            ks.append(kc); vs.append(vc)
        x = F.layer_norm(x, (H,), self.lnfw, self.lnfb, EPS)
        logits = x[:, 0] @ self.wte.T                                       # [1,K]
        return logits, torch.stack(ks), torch.stack(vs)

step_mod = DecoderStep(KEEP).eval()
CACHE_SHAPE = (NL, 1, NH, S_CACHE, DH)
BOS_K = EOS_K                                                               # GPT-2: BOS == EOS

def greedy_steps(prefix, run_step):
    """the app's loop. prefix np [1,P,H]; run_step(prefix, token, pos, k, v) -> (logits, k, v) as numpy. -> (full ids, hit_cap, n_calls)"""
    k = np.zeros(CACHE_SHAPE, np.float32); v = np.zeros(CACHE_SHAPE, np.float32)
    out, tokk, calls = [], 0, 0
    for pos in range(P + MAX_NEW):
        if pos == P:
            tokk = BOS_K
        lg, k, v = run_step(prefix, np.array([tokk], np.int64), np.array([pos], np.int64), k, v)
        calls += 1
        if pos < P:
            continue
        tokk = int(lg[0].argmax())
        if tokk == EOS_K:
            return out, False, calls
        out.append(KEEP[tokk])
        if len(out) == MAX_NEW:
            return out, True, calls
    return out, True, calls

def torch_step(pre, t, p, k, v):
    r = step_mod(torch.from_numpy(pre), torch.from_numpy(t), torch.from_numpy(p), torch.from_numpy(k), torch.from_numpy(v))
    return [x.numpy() for x in r]

# torch check of the hand-written step against model.greedy on a few studies (catches a wrong GPT-2 detail before export)
n_chk = min(8, len(EV["test"]))
same = sum(greedy_steps(ENC_PREFIX[int(i)], torch_step)[0] == REF["test"]["seqs"][j]
           for j, i in enumerate(EV["test"].idx.values[:n_chk]))
log(f"hand-written decoder step (torch) vs model.greedy: {same}/{n_chk} identical")
assert same >= n_chk - 1, "the step module does not reproduce model.greedy - do not export"

DEC32 = f"{BUNDLE}/decoder_fp32.onnx"
_args = (torch.from_numpy(ENC_PREFIX[int(EV['test'].idx.values[0])]), torch.tensor([0]), torch.tensor([0]),
         torch.zeros(CACHE_SHAPE), torch.zeros(CACHE_SHAPE))
onnx_export(step_mod, _args, DEC32, ["prefix", "token", "pos", "k_cache", "v_cache"], ["logits", "k_out", "v_out"])

from onnxruntime.quantization import quantize_dynamic, QuantType
DEC8 = f"{BUNDLE}/decoder_int8.onnx"
quantize_dynamic(DEC32, DEC8, op_types_to_quantize=["MatMul"], per_channel=True, weight_type=QuantType.QInt8,
                 extra_options={"MatMulConstBOnly": True})
_ops = {}
for n in onnx.load(DEC8).graph.node:
    _ops[n.op_type] = _ops.get(n.op_type, 0) + 1
log(f"int8 decoder: {os.path.getsize(DEC8) / 2**20:.1f} MB | ops {dict(sorted(_ops.items()))}")

# Fallback candidates (added after the export-v1 dry run, before the full run). Dynamic INT8 also quantises ACTIVATIONS with
# one scale per tensor; GPT-2's residual stream has a few very large outlier channels, so that scale leaves little precision
# for the rest. Both candidates below keep activations in fp32:
#   A5 "w4"  : 4-bit weight-only, blockwise (32), symmetric (ORT MatMulNBits; has fast CPU kernels; 8-bit had none -> 6x slower than fp32)
#   A6 "f16w": weights stored as fp16, Cast to fp32 when the session loads (halves the file; compute and speed = fp32)
from onnx import numpy_helper, helper, TensorProto
import logging
logging.getLogger("onnxruntime.quantization.matmul_nbits_quantizer").setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)
DECW4 = f"{BUNDLE}/decoder_w4.onnx"
try:
    from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer
    _q = MatMulNBitsQuantizer(onnx.load(DEC32), bits=4, block_size=32, is_symmetric=True)
    _q.process()
    _q.model.save_model_to_file(DECW4, use_external_data_format=False)
    _ops = {}
    for n in onnx.load(DECW4).graph.node:
        _ops[n.op_type] = _ops.get(n.op_type, 0) + 1
    log(f"w4 decoder (4-bit weight-only): {os.path.getsize(DECW4) / 2**20:.1f} MB | MatMulNBits {_ops.get('MatMulNBits', 0)}, MatMul left {_ops.get('MatMul', 0)}")
except Exception as ex:                       # a tooling problem must not stop the run: A5 is then reported as unavailable
    log(f"w4 decoder NOT created ({ex!r}) -> A5 skipped")
    DECW4 = None

DECF16 = f"{BUNDLE}/decoder_f16w.onnx"
_m = onnx.load(DEC32); _g = _m.graph; _new, _casts = [], []
for init in list(_g.initializer):
    a_ = numpy_helper.to_array(init)
    if a_.dtype == np.float32 and a_.size >= 4096:
        _new.append(numpy_helper.from_array(a_.astype(np.float16), init.name + "_fp16"))
        _casts.append(helper.make_node("Cast", [init.name + "_fp16"], [init.name], to=TensorProto.FLOAT))
        _g.initializer.remove(init)
_g.initializer.extend(_new)
_nodes = list(_g.node); del _g.node[:]; _g.node.extend(_casts + _nodes)
onnx.save(_m, DECF16); onnx.checker.check_model(DECF16)
log(f"f16w decoder (fp16 storage): {os.path.getsize(DECF16) / 2**20:.1f} MB")
RES["sizes_mb"] = {os.path.basename(p): round(os.path.getsize(p) / 2**20, 1) for p in (ENC_PATH, DEC32, DEC8, DECW4, DECF16) if p}


In [ ]:
# ---- A3 / A4: decode TEST with the ONNX decoders exactly as the app will
def ort_step_fn(sess):
    def f(pre, t, p, k, v):
        return sess.run(None, {"prefix": pre, "token": t, "pos": p, "k_cache": k, "v_cache": v})
    return f

DEC = {}
CAND = [(n, p_) for n, p_ in (("fp32", DEC32), ("int8", DEC8), ("w4", DECW4), ("f16w", DECF16)) if p_]
for name, path in CAND:
    sess = ort_session(path)
    fn = ort_step_fn(sess)
    t0, seqs, caps, calls = time.time(), [], [], 0
    for i in EV["test"].idx.values:
        s_, c_, n_ = greedy_steps(ENC_PREFIX[int(i)], fn)
        seqs.append(s_); caps.append(c_); calls += n_
    dt = time.time() - t0
    DEC[name] = {"seqs": seqs, "cap": np.array(caps), "text": [decode(q) for q in seqs], "ms_per_step": dt / calls * 1000,
                 "s_per_report": dt / len(seqs)}
    log(f"ORT {name}: {len(seqs)} TEST reports in {dt:.0f} s | {DEC[name]['ms_per_step']:.1f} ms per step (4 threads) | "
        f"{DEC[name]['s_per_report']:.2f} s per report incl. {P} prefix steps")

ident = float(np.mean([a == b for a, b in zip(DEC["fp32"]["seqs"], REF["test"]["seqs"])]))
A3_PASS = ident >= 0.99
log(f"A3 ONNX fp32 vs torch reference: {ident:.1%} identical TEST reports -> {'PASS' if A3_PASS else 'FAIL'}")
RES["checks"]["A3_onnx_fp32_vs_torch"] = {"identical": ident, "pass": A3_PASS}

# clinical metrics (same definitions as the distill notebook: CheXbert(text) vs CheXbert(GT impression), positive = class 1)
def text_tokens(t):
    return re.findall(r"[a-z0-9]+", str(t).lower())
def has_rep3(t):
    w = text_tokens(t); tg = list(zip(w, w[1:], w[2:]))
    return len(tg) != len(set(tg))
J5 = [LABELS.index(c) for c in FIVE]
_all = sorted(set(EV["test"].impression_final) | set(REF["test"]["text"]) | set(DEC["fp32"]["text"]) | set(DEC["int8"]["text"]) | {t for n_ in DEC for t in DEC[n_]["text"]})
_cls = dict(zip(_all, cxb.classes(_all)))
def lab(texts):
    return binary(np.stack([_cls[t] for t in texts]), LABELS, "pos_only").astype(bool)
GT = lab(list(EV["test"].impression_final))
def counts(Pm, Gm):
    return np.stack([(Pm & Gm).sum(1), (Pm & ~Gm).sum(1), (~Pm & Gm).sum(1)], 1)     # per-study tp, fp, fn
def f1(c):
    tp, fp, fn = c.sum(0)
    return float(2 * tp / max(2 * tp + fp + fn, 1))
def summary(texts, cap):
    Pm = lab(texts)
    c14, c5 = counts(Pm, GT), counts(Pm[:, J5], GT[:, J5])
    tp5, fp5, fn5 = c5.sum(0)
    return {"CE14": f1(c14), "CE5": f1(c5), "CE5_recall": float(tp5 / max(tp5 + fn5, 1)), "CE5_precision": float(tp5 / max(tp5 + fp5, 1)),
            "distinct": len(set(texts)), "hit_cap": float(np.mean(cap)), "rep3": float(np.mean([has_rep3(t) for t in texts])),
            "pos_per_study": float(Pm[:, 1:].sum(1).mean()), "_c14": c14, "_c5": c5}
SM = {"reference": summary(REF["test"]["text"], REF["test"]["cap"]),
      "fp32": summary(DEC["fp32"]["text"], DEC["fp32"]["cap"]), "int8": summary(DEC["int8"]["text"], DEC["int8"]["cap"]),
      }
for n_ in ("w4", "f16w"):
    if n_ in DEC:
        SM[n_] = summary(DEC[n_]["text"], DEC[n_]["cap"])
for k, v in SM.items():
    log(f"  {k:9s} CE14 {v['CE14']:.3f} | CE5 {v['CE5']:.3f} | CE5 recall {v['CE5_recall']:.3f} | prec {v['CE5_precision']:.3f} | distinct {v['distinct']} | "
        f"hit-cap {v['hit_cap']:.1%} | rep3 {v['rep3']:.1%} | pos/study {v['pos_per_study']:.2f}")
log("  (distill-v10b reported K10s1_fb2 on TEST with GPU fp16: CE14 0.589, CE5 0.521, CE5 recall 0.485)")

rng = np.random.default_rng(SEED)
n = len(GT)
B = rng.integers(0, n, (N_BOOT, n))
def paired_ci(name, key):
    a, b = SM[name][key], SM["fp32"][key]
    d = [f1(a[ix]) - f1(b[ix]) for ix in B]
    return f1(a) - f1(b), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def quant_rule(name, tag):
    """same pre-registered criteria for every quantised decoder vs ONNX fp32 (A4 dynamic int8, A5 4-bit weight-only, A6 fp16 storage)"""
    d14, d5 = paired_ci(name, "_c14"), paired_ci(name, "_c5")
    sq, s32 = SM[name], SM["fp32"]
    ok = (d14[1] > -0.02 and d5[1] > -0.02 and sq["distinct"] >= 0.9 * s32["distinct"]
          and sq["hit_cap"] <= s32["hit_cap"] + 0.01 and sq["rep3"] <= s32["rep3"] + 0.01)
    same = float(np.mean([a == b for a, b in zip(DEC[name]["seqs"], DEC["fp32"]["seqs"])]))
    log(f"{tag} {name} - fp32 (paired, TEST): CE14 {d14[0]:+.3f} [{d14[1]:+.3f}, {d14[2]:+.3f}] | CE5 {d5[0]:+.3f} [{d5[1]:+.3f}, {d5[2]:+.3f}] | "
        f"distinct {sq['distinct']} vs {s32['distinct']} | identical reports {same:.1%} -> {'PASS' if ok else 'FAIL'}")
    RES["checks"][f"{tag}_{name}"] = {"dCE14": d14, "dCE5": d5, "identical_to_fp32": same, "pass": ok}
    return ok
A4_PASS = quant_rule("int8", "A4")
A5_PASS = quant_rule("w4", "A5") if "w4" in SM else False
A6_PASS = quant_rule("f16w", "A6") if "f16w" in SM else False
RES["test_metrics"] = {k: {kk: vv for kk, vv in v.items() if not kk.startswith("_")} for k, v in SM.items()}
# shipping order (int8 first as pre-registered for A4; A5/A6 added after the dry run): int8 -> w4 -> f16w -> fp32
SHIP, SHIP_PATH = next((n, p_) for ok, n, p_ in ((A4_PASS, "int8", DEC8), (A5_PASS, "w4", DECW4), (A6_PASS, "f16w", DECF16), (True, "fp32", DEC32)) if ok)
RES["shipped_decoder"] = os.path.basename(SHIP_PATH)
log(f"shipped decoder: {SHIP} ({os.path.getsize(SHIP_PATH) / 2**20:.0f} MB)")

# single-thread latency (closer to one phone big core), 20 studies
lat = {}
for name, path in [("encoder", ENC_PATH)] + CAND:
    sess = ort_session(path, threads=1)
    t0 = time.time()
    if name == "encoder":
        for i in EV["test"].idx.values[:20]:
            sess.run(None, {"image": np.ascontiguousarray(imgs[i:i + 1])})
        lat["encoder_ms"] = (time.time() - t0) / 20 * 1000
    else:
        calls = 0
        for i in EV["test"].idx.values[:20]:
            calls += greedy_steps(ENC_PREFIX[int(i)], ort_step_fn(sess))[2]
        lat[f"{name}_ms_per_step"] = (time.time() - t0) / calls * 1000
log(f"Kaggle CPU, 1 thread: {', '.join(f'{k} {v:.1f}' for k, v in lat.items())}")
RES["latency_kaggle_cpu_1thread"] = lat


In [ ]:
# ---- app bundle: vocabulary, config, demo samples
def bytes_to_unicode():                                    # GPT-2 byte-level BPE alphabet (Radford et al. 2019, openai/gpt-2 encoder.py)
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
BYTE_DEC = {c: b for b, c in bytes_to_unicode().items()}
if OFFLINE_TEST:
    pieces = [("" if t == EOS_ID else f" w{t}") for t in KEEP]
else:
    raw = tok.convert_ids_to_tokens(KEEP)
    pieces = ["" if t == EOS_ID else bytes(BYTE_DEC[c] for c in r).decode("utf-8", errors="replace") for t, r in zip(KEEP, raw)]
non_ascii = sum(any(ord(ch) > 127 for ch in p) for p in pieces)
# the app joins pieces and strips: check that this reproduces the tokenizer's decode on every shipped TEST report
_join = lambda s: "".join(pieces[FULL2K[t]] for t in s).strip()
vocab_ok = float(np.mean([_join(s) == t for s, t in zip(DEC[SHIP]["seqs"], DEC[SHIP]["text"])]))
log(f"vocab.json: {len(pieces)} pieces, {non_ascii} non-ASCII | app-side join == tokenizer decode on {vocab_ok:.1%} of TEST reports")
assert vocab_ok > 0.999
json.dump({"pieces": pieces, "gpt2_ids": KEEP, "eos_index": EOS_K, "bos_index": BOS_K}, open(f"{BUNDLE}/vocab.json", "w"))

# per-flag TEST numbers from probe-v2
_pr = [p for p in glob.glob(f"{INPUT_ROOT}/**/probe_results.json", recursive=True) if "table" in json.load(open(p))]
TABLE = {r["label"]: r for r in json.load(open(_pr[-1]))["table"]} if _pr else {}
def _first(s):
    try:
        return float(str(s).split()[0])
    except Exception:
        return None
flags = []
for j, c in enumerate(LABELS):
    if c not in PJ["in_app"]:
        continue
    r = TABLE.get(c, {})
    flags.append({"label": c, "index": j, "display": PJ["display_text"][c], "threshold_logit": float(THR[j]),
                  "test_sensitivity": _first(r.get(f"sens@{APP_TARGET}")), "test_sensitivity_ci": r.get(f"sens@{APP_TARGET}"),
                  "test_ppv": r.get(f"PPV@{APP_TARGET}"), "test_specificity": r.get(f"spec@{APP_TARGET}"),
                  "test_auc": r.get("test_AUC"), "text_sensitivity": r.get("text_sens"), "test_positives": r.get("test_pos")})
never = [f["label"] for f in flags if f["text_sensitivity"] == 0]
sm = SM[SHIP]
CONFIG = {
    "version": EXPORT_VERSION, "run": RUN, "probe": PJ.get("version"),
    "encoder": {"file": "encoder.onnx", "input": "image", "input_dtype": "uint8", "input_shape": [1, 512, 512],
                "outputs": {"prefix": [1, P, gc.n_embd], "probe_logits": [1, len(LABELS)], "features": [1, 2 * model.d_v]},
                "note": "384 px resize and 0.5/0.5 normalisation are inside the graph"},
    "app_preprocessing": "IE643 Cleaning step 5: decode to 8-bit grayscale; scale so the LONGER side is 512 with area interpolation (cv2.INTER_AREA = average of the covered source pixels); new size = round(h*s), round(w*s); paste centred on a black 512x512 canvas at ((512-nh)//2, (512-nw)//2)",
    "decoder": {"file": os.path.basename(SHIP_PATH), "precision": SHIP, "inputs": ["prefix", "token", "pos", "k_cache", "v_cache"],
                "outputs": ["logits", "k_out", "v_out"], "cache_shape": list(CACHE_SHAPE), "n_prefix": P, "max_new": MAX_NEW,
                "vocab_size": K, "eos_index": EOS_K, "bos_index": BOS_K,
                "loop": "pos 0..n_prefix-1: feed prefix (token ignored); pos n_prefix: token = bos_index; then feed argmax back; "
                        "stop at eos_index or after max_new tokens; pass k_out/v_out back as k_cache/v_cache"},
    "labels": LABELS, "flags": flags, "flag_target_sensitivity": float(APP_TARGET),
    "not_assessed": PJ["not_assessed"], "text_never_names": never, "limits": PJ["limits"],
    "metrics_this_export_test": {k: sm[k] for k in ("CE14", "CE5", "CE5_recall", "CE5_precision", "distinct")} | {"n": len(GT)},
    "metrics_probe_v2_test": PJ.get("test_summary"),
    "metrics_eval_final_v2_test": {
        "source": "eval-final-v2 rerun with the v10b output (2026-10-24); refresh from eval-final-v4",
        "K10s1_fb2": {"RG_ER": 0.146, "BERTScore": 0.353, "RadCliQ_v1": 1.262, "CE5_recall": 0.485, "CE5_specificity": 0.930,
                      "false_reassurance": 0.278},
        "teacher_CheXagent2_3b": {"CE14": 0.450, "CE5": 0.376, "CE5_recall": 0.303, "RG_ER": 0.105, "RadCliQ_v1": 1.414,
                                  "false_reassurance": 0.257}},
}
json.dump(CONFIG, open(f"{BUNDLE}/app_config.json", "w"), indent=1)
log(f"app_config.json: {len(flags)} flags, text never names {never}")

# demo samples: TEST studies, half with one of the five common findings, a quarter normal, rest random
from PIL import Image
rng = np.random.default_rng(SEED)
ti = EV["test"].idx.values
ab = np.where(GT[:, J5].any(1))[0]; nm = np.where(GT[:, 0])[0]
pick = list(rng.choice(ab, min(len(ab), N_SAMPLES // 2), replace=False))
pick += list(rng.choice(np.setdiff1d(nm, pick), min(len(np.setdiff1d(nm, pick)), N_SAMPLES // 4), replace=False))
rest = np.setdiff1d(np.arange(len(ti)), pick)
pick += list(rng.choice(rest, min(len(rest), N_SAMPLES - len(pick)), replace=False))
samples = []
for j in pick:
    i = int(ti[j]); fn = f"test_{i}.png"
    Image.fromarray(np.asarray(imgs[i])).save(f"{BUNDLE}/samples/{fn}")
    lg = ENC_LOGITS[j]
    samples.append({"file": fn, "idx": i, "gt_impression": EV["test"].impression_final[j], "model_impression": DEC[SHIP]["text"][j],
                    "gt_positive": [c for c, v in zip(LABELS, GT[j]) if v],
                    "flags": [f["label"] for f in flags if lg[f["index"]] >= f["threshold_logit"]]})
json.dump(samples, open(f"{BUNDLE}/samples/samples.json", "w"), indent=1)
log(f"samples: {len(samples)} TEST images -> app_bundle/samples (Stanford RUA: keep the app private, do not publish the APK)")

RES["checks_passed"] = {k: v.get("pass") for k, v in RES["checks"].items() if isinstance(v, dict) and "pass" in v}
RES["partial"] = N_EVAL is not None
RES["minutes"] = round((time.time() - T0) / 60, 1)
json.dump(RES, open(f"{WORK}/export_results.json", "w"), indent=1, default=float)
import shutil
for p in [q for q in (DEC32, DEC8, DECW4, DECF16) if q and q != SHIP_PATH and os.path.exists(q)]:
    shutil.move(p, f"{WORK}/{os.path.basename(p)}")      # keep only the shipped decoder inside the bundle
shutil.make_archive(f"{WORK}/app_bundle", "zip", BUNDLE)
log(f"\nbundle: {sorted(os.listdir(BUNDLE))} | zip {os.path.getsize(f'{WORK}/app_bundle.zip') / 2**20:.0f} MB")
log(f"checks: {RES['checks_passed']} | shipped decoder: {SHIP} | total {RES['minutes']} min")
print("EXPORT OK")
